# multi_strat_045

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (59).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 7 |
| Detected functions | — |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/ipynb/HVStocks/RELIANCE_minute.csv')
data

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from datetime import timedelta

# Load data (replace 'price_data.csv' with your file name)
data.set_index("date", inplace=True)

# Ensure the index is monotonic and handle duplicates
if not data.index.is_monotonic_increasing:
    data = data.sort_index()

if data.index.duplicated().any():
    data = data[~data.index.duplicated(keep='first')]

# Holt-Winters model configuration
seasonal_periods = 60  # Adjust based on data frequency
model = ExponentialSmoothing(data['close'], trend="add", seasonal="add", seasonal_periods=seasonal_periods)
hw_fit = model.fit()

# Add Holt-Winters components to the DataFrame
data['forecast'] = hw_fit.fittedvalues
data['trend'] = hw_fit.level + hw_fit.trend
data['seasonality'] = hw_fit.season

# Preallocate arrays for tradebook
tradebook = []
capital = 100000  # Fixed capital per trade
brokerage_rate = 0.001  # 0.1% brokerage
cumulative_pnl = 0  # Cumulative PnL

# Initialize position
position = None
entry_price = 0
entry_time = None

# Vectorize trend difference calculation
data['trend_diff'] = data['trend'].diff().shift(-1)  # Calculate trend difference for each row

# Vectorized trade decisions
buy_signal = (data['close'] < data['forecast']) & (data['trend_diff'] > 0)
sell_signal = (data['close'] > data['forecast']) & (data['trend_diff'] < 0)

# Track trades
for i in range(1, len(data)):
    if position is None:
        if buy_signal.iloc[i]:  # Buy signal
            position = "long"
            entry_price = data['close'].iloc[i]
            entry_time = data.index[i]
        elif sell_signal.iloc[i]:  # Sell signal
            position = "short"
            entry_price = data['close'].iloc[i]
            entry_time = data.index[i]
    elif position == "long" or position == "short":
        # Close position at 15:28 or based on reverse signal
        if data.index[i].time() >= pd.Timestamp("15:28").time() or \
                (position == "long" and sell_signal.iloc[i]) or \
                (position == "short" and buy_signal.iloc[i]):
            exit_price = data['close'].iloc[i]
            pnl = capital * (exit_price - entry_price) / entry_price if position == "long" else \
                capital * (entry_price - exit_price) / entry_price
            brokerage = brokerage_rate * capital
            net_pnl = pnl - brokerage
            cumulative_pnl += net_pnl

            # Record trade in tradebook
            tradebook.append({
                "Entry Time": entry_time,
                "Exit Time": data.index[i],
                "Entry Price": entry_price,
                "Exit Price": exit_price,
                "Action": "Buy" if position == "long" else "Sell",
                "PnL": net_pnl,
                "Cumulative PnL": cumulative_pnl
            })

            # Reset position
            position = None
            entry_price = 0
            entry_time = None

# Convert tradebook to DataFrame
tradebook_df = pd.DataFrame(tradebook)

# Save tradebook to CSV
tradebook_df.to_csv("tradebook.csv", index=False)

# Print summary
print("Backtesting Completed!")
print(tradebook_df.tail())


In [ ]:
tradebook_df

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Assuming 'data' is your DataFrame with the necessary columns
# For demonstration, we'll use the first 10,000 rows
data_subset = data.iloc[:10000]

# Create traces for the plot
trace_close = go.Scatter(
    x=data_subset.index,
    y=data_subset['close'],
    mode='lines',
    name='Close Price',
    line=dict(color='blue')
)

trace_trend = go.Scatter(
    x=data_subset.index,
    y=data_subset['trend'],
    mode='lines',
    name='Trend',
    line=dict(color='orange')
)

# Identify buy and sell signals
buy_signals = tradebook_df[tradebook_df['Action'] == 'Buy']
sell_signals = tradebook_df[tradebook_df['Action'] == 'Sell']

trace_buys = go.Scatter(
    x=buy_signals.index,
    y=buy_signals,
    mode='markers',
    name='Buy Signals',
    marker=dict(symbol='triangle-up', color='green', size=10)
)

trace_sells = go.Scatter(
    x=sell_signals.index,
    y=sell_signals,
    mode='markers',
    name='Sell Signals',
    marker=dict(symbol='triangle-down', color='red', size=10)
)

# Create the figure
fig = go.Figure(data=[trace_close, trace_trend, trace_buys, trace_sells])

# Update layout
fig.update_layout(
    title='Close Price, Trend, and Trading Signals',
    xaxis_title='Date',
    yaxis_title='Price',
    legend_title='Legend',
    template='plotly_dark'
)

# Show the plot
fig.show()
